<a href="https://colab.research.google.com/github/AbdelrahmanMohamed-Fathy/compilers-assignment-2/blob/main/Compilers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Input and Verification


In [453]:
from pydantic import BaseModel, Field, field_serializer, computed_field
from typing import Optional, ClassVar
from enum import Enum

# @title reading regex string
regex = input("enter your regex: ")

In [454]:
# @title function to return the opposite pair of a bracket
def get_pair(bracket: str) -> str:
    match bracket:
        case ")":
            return "("
        case "]":
            return "["
        case _:
            raise ValueError("Received none bracket")

In [455]:
# @title check for correct brackets using a stack
stack = []
for char in regex:
    # adding to stack if opening a bracket
    if char == "(" or char == "[":
        stack.append(char)
    # popping from stack and checking if matching bracket pair
    elif char == ")" or char == "]":
        if stack.pop() != get_pair(char):
            raise ValueError("Invalid regex")

if len(stack) != 0:
    raise ValueError("Invalid regex")

In [ ]:
# @title Model for regex groups


class GroupModifier(Enum):
    PLUS = "+"
    STAR = "*"


class GroupType(Enum):
    SEQUENCE = 0
    OR = 1


class RegexGroup(BaseModel):
    tokens: list["str | RegexGroup"] = Field(default=[])
    type: GroupType = Field(default=GroupType.SEQUENCE)
    modifier: Optional[GroupModifier] = Field(default=None)

    def insert(self, token: str):
        self.tokens.append(token)

    def insert_group(self, group: "RegexGroup"):
        self.tokens.append(group)


In [ ]:
# @title Seperating regex into tokens


def parse_regex(regex: str) -> RegexGroup:
    token_groups: RegexGroup = RegexGroup()
    parse_index: int = 0
    while parse_index < len(regex):
        match regex[parse_index]:
            # sequence group handling
            case "(":
                token_groups.insert_group(parse_regex(regex[parse_index + 1 :]))
                parse_index = regex.find(")", parse_index) + 1
                # skip plus and star related to group
                if parse_index < len(regex) and regex[parse_index] in {
                    "+",
                    "*",
                }:
                    parse_index += 1

            case ")":
                if parse_index + 1 < len(regex) and regex[parse_index + 1] == "+":
                    token_groups.modifier = GroupModifier.PLUS
                elif parse_index + 1 < len(regex) and regex[parse_index + 1] == "*":
                    token_groups.modifier = GroupModifier.STAR
                return token_groups
            # or group handling
            case "[":
                token_groups.insert_group(parse_regex(regex[parse_index + 1 :]))
                parse_index = regex.find("]", parse_index) + 1
                # skip plus and star related to group
                if parse_index < len(regex) and regex[parse_index] in {
                    "+",
                    "*",
                }:
                    parse_index += 1
            case "]":
                token_groups.type = GroupType.OR
                if parse_index + 1 < len(regex) and regex[parse_index + 1] == "+":
                    token_groups.modifier = GroupModifier.PLUS
                elif parse_index + 1 < len(regex) and regex[parse_index + 1] == "*":
                    token_groups.modifier = GroupModifier.STAR
                return token_groups
            # literals
            case _:
                if parse_index + 1 < len(regex) and regex[parse_index + 1] == "+":
                    token_groups.insert_group(
                        RegexGroup(
                            tokens=[regex[parse_index]], modifier=GroupModifier.PLUS
                        )
                    )
                    parse_index += 2
                elif parse_index + 1 < len(regex) and regex[parse_index + 1] == "*":
                    token_groups.insert_group(
                        RegexGroup(
                            tokens=[regex[parse_index]], modifier=GroupModifier.STAR
                        )
                    )
                    parse_index += 2
                elif parse_index + 1 < len(regex) and regex[parse_index + 1] == "-":
                    token_groups.insert(regex[parse_index : parse_index + 3])
                    parse_index += 3
                else:
                    token_groups.insert(regex[parse_index])
                    parse_index += 1

    return token_groups


token_list = parse_regex(regex)
print(token_list)

tokens=['k', RegexGroup(tokens=['O'], type=<GroupType.SEQUENCE: 0>, modifier=<GroupModifier.PLUS: '+'>), '=', '=', '=', '=', 'F', RegexGroup(tokens=['f'], type=<GroupType.SEQUENCE: 0>, modifier=<GroupModifier.STAR: '*'>), 'a', 'a', 'a'] type=<GroupType.SEQUENCE: 0> modifier=None


# Generating NFA


In [458]:
# @title Regex model class as a tree node


class RegexTreeNode(BaseModel):
    _name_counter: ClassVar[int] = 0
    name: str = Field(default="Unnamed")
    is_terminating_state: bool = Field(serialization_alias="isTerminatingState")
    children: Optional[dict[str, list["RegexTreeNode"]]] = Field(
        serialization_alias="paths",
        exclude_if=lambda paths: not paths,
    )

    def __init__(self, **data):
        super().__init__(**data)
        self.name = f"S{RegexTreeNode._name_counter}"
        RegexTreeNode._name_counter += 1
        self.is_terminating_state = data["is_terminating_state"]
        self.children = data["children"]

    @field_serializer("children")
    def serialize_children(self, children: dict[str, list["RegexTreeNode"]], _info):
        return {k: [node.name for node in v] for k, v in children.items()}


In [459]:
# @title NFA model class as a tree structure


class NfaModel(BaseModel):
    nodes: list[RegexTreeNode] = Field(default=[], serialization_alias="States")

    def insert(self, node: RegexTreeNode):
        self.nodes.append(node)

    @computed_field(alias="startingState")
    @property
    def starting_node(self) -> str:
        return self.nodes[0].name


In [460]:
# @title Parsing functions


def or_case_handler(
    nfa: NfaModel, regex_list: list[str], is_terminating: bool = False
) -> RegexTreeNode:
    initial_node = RegexTreeNode(is_terminating_state=False, children={"Ɛ": []})
    final_node = RegexTreeNode(is_terminating_state=is_terminating, children=None)
    nfa.insert(initial_node)
    # handle each indivisually and append it to the initial node while passing the final node as the child to all of them
    for regex in regex_list:
        initial_node.children["Ɛ"].append(
            base_case_handler(nfa=nfa, regex=regex, child=("Ɛ", [final_node]))
        )
    nfa.insert(final_node)
    return initial_node


# handling base case of 1 node going to a second node
def base_case_handler(
    nfa: NfaModel,
    regex: str,
    is_terminating: bool = False,
    child: tuple[str, list[str]] | None = None,
) -> RegexTreeNode:
    second = RegexTreeNode(
        is_terminating_state=is_terminating,
        children={child[0]: child[1]} if child else None,
    )
    first = RegexTreeNode(
        is_terminating_state=False,
        children={regex: [second]},
    )
    nfa.insert(first)
    nfa.insert(second)
    return first


def plus_case_handler(is_star: bool):
    pass

In [461]:
# @title test output for (a-z|A-Z|/)

nfa: NfaModel = NfaModel()
or_case_handler(nfa=nfa, regex_list=["a-z", "A-Z", "/"], is_terminating=True)

print(nfa.model_dump_json(by_alias=True, indent=2))

{
  "States": [
    {
      "name": "S0",
      "isTerminatingState": false,
      "paths": {
        "Ɛ": [
          "S3",
          "S5",
          "S7"
        ]
      }
    },
    {
      "name": "S3",
      "isTerminatingState": false,
      "paths": {
        "a-z": [
          "S2"
        ]
      }
    },
    {
      "name": "S2",
      "isTerminatingState": false,
      "paths": {
        "Ɛ": [
          "S1"
        ]
      }
    },
    {
      "name": "S5",
      "isTerminatingState": false,
      "paths": {
        "A-Z": [
          "S4"
        ]
      }
    },
    {
      "name": "S4",
      "isTerminatingState": false,
      "paths": {
        "Ɛ": [
          "S1"
        ]
      }
    },
    {
      "name": "S7",
      "isTerminatingState": false,
      "paths": {
        "/": [
          "S6"
        ]
      }
    },
    {
      "name": "S6",
      "isTerminatingState": false,
      "paths": {
        "Ɛ": [
          "S1"
        ]
      }
    },
    {
      "name

# Optimizing into DFA
